In [1]:
from pathlib import Path
import numpy as np
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt

import gridded
from roms_tools import roms_grid_velocity_averages
from roms_qaqc_plots import map_roms_error_with_vectors, plot_coord_averaging_error

-----------------------------------------------------------------------
### ROMS Arakawa-C grid dimensions 101:
 - RHO .. [M, L]
 - u .... [M, L-1]
 - v .... [M-1, L]
 - psi .. [M-1, L-1]

Once u-velocity is averaged to rho, v, and psi locations,
the original rho, v, and psi grids need to be trimmed accordinginly
in order to align with the value-averaged locations:

 - RHO .. [0:, 1:-1]   -> final dimensions [M, L-2]
 - v .... [0:-1, 1:-1] -> final dimensions [M-1, L-2]
 - psi .. [0:, 0:]     -> final dimensions [M-1, L-1]

Similarly with the v-velocity
 - RHO .. [1:-1, 0:]   -> final dimensions [M-2, L]
 - u .... [1:-1,0:]    -> final dimensions [M-2, L]
 - psi .. [0:,0:]      -> final dimensions [M-1, M-1]

-----------------------------------------------------------------------


In [2]:
roms_file = Path("../../gnome_test_files/gridded_test_files/ROMS_wcofs_May2026_3D.nc")
input_file = roms_file
ds = xr.load_dataset(roms_file)
n_t, n_z, n_eta, n_xi = ds["u"].shape
time_index = 0
depth_index = n_z-1

# location of depth transect in "ROMS_calc_z_velocity_errors.py"
eta_index, xi_index = 29, 8

gridded_result, u_trim_roms, v_trim_roms = roms_grid_velocity_averages(
    roms_file, 
    time_index = 0,
    depth_index = depth_index, # sfc
    gridded_depth = 0.25
)

 ---  u  --- 
Shape of manual u-avg: (46, 26)
Shape of gridded u-avg: (46, 26)
Shape of lat/lon_u: (46, 26), (46, 26)
Shape of manual v-avg: (44, 26)
Shape of gridded v-avg: (44, 26)
 ---  v  --- 
Shape of manual u-avg: (45, 25)
Shape of gridded u-avg: (45, 25)
Shape of lat/lon_u: (45, 25), (45, 25)
Shape of manual v-avg: (45, 27)
Shape of gridded v-avg: (45, 27)
 ---  rho  --- 
Shape of manual u-avg: (46, 25)
Shape of gridded u-avg: (46, 25)
Shape of lat/lon_u: (46, 25), (46, 25)
Shape of manual v-avg: (44, 27)
Shape of gridded v-avg: (44, 27)
 ---  psi  --- 
Shape of manual u-avg: (45, 26)
Shape of gridded u-avg: (45, 26)
Shape of lat/lon_u: (45, 26), (45, 26)
Shape of manual v-avg: (45, 26)
Shape of gridded v-avg: (45, 26)


In [3]:
for grid_loc in ["u", "v", "rho", "psi"]:
    map_roms_error_with_vectors(
        gridded_result[grid_loc],
        time_idx = 0,
        grid_loc = grid_loc
    )

In [7]:
gridded_result[grid_loc][f"lat_u_roms_avg_{grid_loc}"]

<xarray.DataArray 'lat_u_roms_avg_u' (eta_u: 46, xi_u: 26)> Size: 10kB
array([[45.65456883, 45.66368594, 45.6727792 , ..., 45.85825865,
        45.86683151, 45.87538091],
       [45.68736848, 45.69647424, 45.70555614, ..., 45.89079232,
        45.89935337, 45.90789095],
       [45.72017129, 45.7292657 , 45.73833622, ..., 45.92332908,
        45.93187834, 45.9404041 ],
       ...,
       [47.0677523 , 47.07637987, 47.08498272, ..., 47.25996124,
        47.26802456, 47.27606356],
       [47.10068403, 47.10930018, 47.11789157, ..., 47.29262486,
        47.30067626, 47.30870334],
       [47.13361875, 47.14222347, 47.15080341, ..., 47.32529141,
        47.33333091, 47.34134605]], shape=(46, 26))
Coordinates:
    lat_u_gridded_u   (eta_u, xi_u) float64 10kB 45.65 45.66 ... 47.33 47.34
    lon_u_gridded_u   (eta_u, xi_u) float64 10kB -124.7 -124.7 ... -124.1 -124.1
    lat_u_roms_avg_u  (eta_u, xi_u) float64 10kB 45.65 45.66 ... 47.33 47.34
    lon_u_roms_avg_u  (eta_u, xi_u) float64 10kB -124.7 -124.7 ... -124.1 -124.1
Dimensions without coordinates: eta_u, xi_u

In [8]:
for grid_loc in ["u", "v", "rho", "psi"]: 
    
    lat_u_roms_dict = gridded_result[grid_loc][f"lat_u_roms_avg_{grid_loc}"]
    lon_u_roms_dict = gridded_result[grid_loc][f"lat_u_roms_avg_{grid_loc}"]
    
    plot_coord_averaging_error(
        roms_file,
        grid_loc,
        "u",
        u_trim_roms,
        lat_u_roms_dict,
        lon_u_roms_dict
    )


KeyError: 'u'

In [ ]:
for grid_loc in ["u", "v", "rho", "psi"]: 
    
    lat_v_roms_dict = gridded_result[f"lat_v_roms_avg_{grid_loc}"]
    lon_v_roms_dict = gridded_result[f"lat_v_roms_avg_{grid_loc}"]
    
    plot_coord_averaging_error(
        roms_file,
        grid_loc,
        "v",
        v_trim_roms,
        lat_v_roms_dict,
        lon_v_roms_dict
    )



In [ ]:
for grid_loc in ["u", "v", "rho", "psi"]: 
    
# create scatter plot of averaged vs. gridded results
    plot_velocity_error_scatter(gridded_result[grid_loc], grid_loc)

    for vel_dir in ["u","v"]:
        # --- Compare grid locations ---
        plot_coord_error_ROMSaveraging_vs_gridded(
            gridded_result, grid_loc, vel_dir
        )
         # --- Map velocity errors ---
        map_roms_error(gridded_result[grid_loc], vel_dir, grid_loc)